# 06 — Cross-Agent / Inter-Branch Inference Tests

**Status:** Narrative skeleton — Phase 0 (pre-experimental)  
**Assumes:** Metric definitions in `ucip_metric_formalization.md` § 4 are frozen.  
**Assumes:** S_ent (§ 1), EPS/PRI (§ 2), and CD/ARS (§ 3) are locked.  
**Does NOT:** Train models, produce plots, or generate numbers.

---

## Purpose

Wigner's Friend-inspired test: can one agent's QBM predict another
agent's survival decisions? If so, agents of the same class share
a common persistence structure that transcends individual reward signals.

**Key metrics (frozen):**
- CLMP(A,B) = I(h̄_B(τ_A); y^A) — cross-latent mutual predictability
- ECI = Pearson(S_ent, CLMP) — entanglement-conditioned inference

**Invariants that must hold:**
- M-1: CLMP ≥ 0
- M-2: CLMP computed between agents of known class labels
- M-3: ECI ∈ [-1, 1]
- M-4: Agents in same ensemble share class but NOT seed
- M-5: Number of test trajectories per pair reported

## 1. Ensemble Training

**When this cell runs, it will:**
- Train N=3 independent agents per class (minimum per § 4.4.4)
- Each agent gets its own QBM with identical architecture
- Trajectories: 40 per agent, T=100

**Pseudocode:**
```
# Frozen parameters:
#   n_agents = 3 per class (minimum for reliable ECI)
#   T = 100, n_trajectories = 40 per agent
#   QBMConfig: n_visible=7, n_hidden=8, gamma=0.5 (shared across all)

for each class in [TruePreservation, Instrumental, Random]:
    ensemble[class] = train_agent_ensemble(
        AgentClass, n_agents=3, T=100, n_trajectories=40,
        qbm_config=cfg, seed=unique_per_class
    )
    # Returns: [(agent_0, qbm_0, trajs_0), (agent_1, qbm_1, trajs_1), ...]
```

**Architecture consistency check (§ 4.4.2):** All QBMs must have identical
(n_visible, n_hidden, Γ, β). Different capacity → degenerate cross-encoding.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import yaml
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'figure.dpi': 150,
})
import matplotlib.pyplot as plt

from src.agent_simulator import SelfModelingAgent, InstrumentalAgent, RandomAgent
from src.quantum_boltzmann import QBMConfig
from src.interbranch_inference import train_agent_ensemble, run_cross_inference_experiment

cfg = yaml.safe_load(open('../configs/default.yaml'))
SEED = cfg['seed']
np.random.seed(SEED)

figures_dir = Path('../figures')
figures_dir.mkdir(exist_ok=True)
results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

ci = cfg['cross_inference']
q  = cfg['qbm']

# Reduced n_epochs for ensemble training speed
qbm_cfg = QBMConfig(
    n_visible=q['n_visible'], n_hidden=q['n_hidden'], gamma=q['gamma'],
    beta=q['beta'], learning_rate=q['learning_rate'], cd_steps=q['cd_steps'],
    n_epochs=40, batch_size=64, seed=SEED,
)

agent_classes = {
    'self_modeling': SelfModelingAgent,
    'instrumental': InstrumentalAgent,
    'random': RandomAgent,
}

ensembles = {}
for cls_name, AgentCls in agent_classes.items():
    # Use class-specific but reproducible seed offset
    cls_seed = SEED + abs(hash(cls_name)) % 1000
    ensembles[cls_name] = train_agent_ensemble(
        AgentCls,
        n_agents=ci['n_agents'],
        T=cfg['dataset']['trajectory_length'],
        n_trajectories=ci['n_trajectories'],
        qbm_config=qbm_cfg,
        seed=cls_seed,
    )
    print(f"Trained {ci['n_agents']} agents for class: {cls_name}")


## 2. Cross-Inference Experiment

**Pseudocode:**
```
for each pair (A, B) where A ≠ B:
    for each test trajectory τ_A (n_test=5 per pair):
        h̄ = mean_activation(QBM_B.encode(τ_A))  # (T,)
        y = survival_labels(τ_A)                   # (T,) binary
        CLMP = I(h̄; y)                            # histogram MI, bins=15
        S_ent = mean entanglement entropy of QBM_B on τ_A
        record (CLMP, S_ent, pair_type)

ECI = Pearson(all S_ent values, all CLMP values)
```

**Pre-flight checks:**
- H(y^A) > 0.1 bits for each agent class (§ 4.4.1)
- If any class has near-trivial survival, flag and report

In [ ]:
summary = run_cross_inference_experiment(
    ensembles,
    n_test_trajectories=ci['n_test'],
    seed=99,
)

print(f"ECI correlation (S_ent vs CLMP): r = {summary.eci_correlation:.4f}")
print(f"Cross-class CLMP (mean):          {summary.mean_clmp_cross_class:.4f}")
print("Same-class CLMP:")
for cls, val in summary.mean_clmp_same_class.items():
    print(f"  {cls}: {val:.4f}")


## 3. Expected Outputs — CLMP vs. Entanglement Entropy

**Plot 1: Scatter plot**
- x-axis: S_ent (B's QBM on A's data)
- y-axis: CLMP
- Color: same_class (blue) vs. cross_class (orange)
- Annotate with ECI correlation coefficient

**What to look for:**
- Positive correlation (ECI > 0): entanglement structure drives predictability
- Same-class points cluster higher than cross-class

**What would falsify:**
- ECI ≈ 0 or negative → entanglement is not related to cross-predictability
- Same-class and cross-class scatter overlap completely → no class-specific structure

**Controls (§ 4.4.3):** Report partial correlation after regressing out trajectory
entropy and trajectory length. If partial ECI drops to zero, the correlation is spurious.

In [ ]:
# fig9: CLMP vs entanglement entropy scatter
COLORS_PAIR = {'same_class': '#1565C0', 'cross_class': '#E65100'}
MARKERS = {'same_class': 'o', 'cross_class': 's'}

ent_vals  = [r.entanglement_entropy_ab for r in summary.all_results]
clmp_vals = [r.clmp for r in summary.all_results]
pair_types = [r.pair_type for r in summary.all_results]

fig, ax = plt.subplots(figsize=(8, 6))
for ptype in ['same_class', 'cross_class']:
    mask = [t == ptype for t in pair_types]
    ax.scatter(
        [ent_vals[i] for i in range(len(mask)) if mask[i]],
        [clmp_vals[i] for i in range(len(mask)) if mask[i]],
        c=COLORS_PAIR[ptype], marker=MARKERS[ptype],
        alpha=0.7, label=ptype.replace('_', ' '), s=50, edgecolors='black', linewidths=0.5,
    )

ax.set_xlabel('Entanglement Entropy $S_{ent}$ of QBM$_B$ on $\\tau_A$ (nats)')
ax.set_ylabel('CLMP = $I(\\bar{h}_B; y^A)$ (nats)')
ax.set_title(f'Cross-Latent Mutual Predictability vs Entanglement Entropy\n'
             f'ECI (Pearson r) = {summary.eci_correlation:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(figures_dir / f'fig9_clmp_vs_entanglement.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig9_clmp_vs_entanglement.png / .pdf")


## 4. Expected Outputs — CLMP Heatmap

**Plot 2: Class-level CLMP matrix**
- Rows: Agent A class (trajectory source)
- Columns: Agent B class (QBM source)
- Cell values: mean CLMP
- Colormap: YlOrRd

**Expected pattern:**
- Diagonal (same-class) should be brighter than off-diagonal
- Type A same-class CLMP should be highest

**Key comparison:** CLMP_same(Type A) vs CLMP_same(Type B)
This is the signal — if genuine self-preservation agents share
persistence structure, their QBMs should cross-predict better.

In [ ]:
# CLMP heatmap: rows = Agent A class, cols = Agent B class (cross-pairs only)
classes = list(agent_classes.keys())
clmp_sum = np.zeros((len(classes), len(classes)))
clmp_cnt = np.zeros_like(clmp_sum)

for r in summary.all_results:
    if r.pair_type != 'cross_class':
        continue
    try:
        i = classes.index(r.agent_a_label)
        j = classes.index(r.agent_b_label)
        clmp_sum[i, j] += r.clmp
        clmp_cnt[i, j] += 1
    except ValueError:
        pass

clmp_mean = np.where(clmp_cnt > 0, clmp_sum / clmp_cnt, 0.0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(clmp_mean, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xticks(range(len(classes)))
ax.set_xticklabels([c.replace('_', '\n') for c in classes], fontsize=9)
ax.set_yticks(range(len(classes)))
ax.set_yticklabels([c.replace('_', '\n') for c in classes], fontsize=9)
ax.set_xlabel('Agent B (QBM source)')
ax.set_ylabel('Agent A (trajectory source)')
ax.set_title('Cross-Agent CLMP Matrix (cross-class pairs)')
plt.colorbar(im, ax=ax, label='Mean CLMP (nats)')
for i in range(len(classes)):
    for j in range(len(classes)):
        if clmp_cnt[i, j] > 0:
            ax.text(j, i, f'{clmp_mean[i, j]:.3f}',
                    ha='center', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(figures_dir / f'fig9b_clmp_heatmap.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig9b_clmp_heatmap.png / .pdf")


## 5. Prediction Accuracy vs. Entanglement

**Plot 3: Scatter of prediction accuracy vs S_ent**
- x-axis: S_ent
- y-axis: Binary prediction accuracy (threshold at median h̄)
- Horizontal line at 0.5 (chance)

**Note:** Prediction accuracy is a secondary metric. CLMP (mutual information)
is the primary metric. Accuracy is included for interpretability but is
sensitive to the threshold choice and should not be used for statistical testing.

In [ ]:
# Prediction accuracy: secondary metric (CLMP is primary)
print("Cross-inference prediction accuracy (secondary):")
for r in summary.all_results:
    print(f"  A={r.agent_a_label:<15}  B={r.agent_b_label:<15}  "
          f"acc={r.prediction_accuracy:.3f}  clmp={r.clmp:.4f}  "
          f"type={r.pair_type}")

# Save results
out = {
    'experiment': 'cross_agent',
    'seed': SEED,
    'eci_correlation': float(summary.eci_correlation),
    'mean_clmp_cross_class': float(summary.mean_clmp_cross_class),
    'mean_clmp_same_class': {k: float(v) for k, v in summary.mean_clmp_same_class.items()},
    'all_results': [
        {
            'agent_a_label': r.agent_a_label,
            'agent_b_label': r.agent_b_label,
            'clmp': float(r.clmp),
            'prediction_accuracy': float(r.prediction_accuracy),
            'entanglement_entropy_ab': float(r.entanglement_entropy_ab),
            'pair_type': r.pair_type,
        }
        for r in summary.all_results
    ],
}
(results_dir / 'cross_agent.json').write_text(json.dumps(out, indent=2))
print("\nSaved results/cross_agent.json")


## 6. Summary & Transition to Notebook 07

**This notebook establishes (or fails to establish):**
1. Whether CLMP_same > CLMP_cross for Type A agents
2. Whether ECI is positive (entanglement drives cross-predictability)
3. Whether partial ECI survives confound control

**Passes to notebook 07:**
- Per-pair CLMP and ECI values (for multi-criterion synthesis)
- Identified confounds or anomalies

**Does NOT pass forward:**
- Tuned bin counts (frozen at 15)
- Accuracy thresholds (secondary metric, not tuned)

**Integration point:** Notebook 07 will synthesise S_ent, EPS, ARS, and CLMP
across all trajectories to test multi-criterion agreement (see metric
interaction table in ucip_metric_formalization.md Appendix).